In [1]:
!pip install -q openai

In [2]:
import os
import re
import time
import pandas as pd
from tqdm import tqdm
import re
from sklearn.metrics import classification_report, accuracy_score

from kaggle_secrets import UserSecretsClient
from openai import OpenAI

In [3]:
# Load OpenAI key
try:
    user_secrets = UserSecretsClient()
    OPENAI_API_KEY = user_secrets.get_secret("OPENAI_API_KEY")
except:
    raise RuntimeError("OPENAI_API_KEY not found")

client = OpenAI(api_key=OPENAI_API_KEY)

In [4]:
class Config:
    MODEL_NAME = "gpt-4.1-2025-04-14"

    POS_DATA_FILE = "/kaggle/input/lrec-tcs-attack-support/sentencePair.txt"
    NEG_DATA_FILE = "/kaggle/input/lrec-tcs-attack-support/sentencePair_neg.txt"
    OUTPUT_FILE = "/kaggle/working/openai_fewshot_cot.csv"

    SAMPLE_SIZE = 500
    RANDOM_SEED = 42

    TEMPERATURE = 0.0
    MAX_OUTPUT_TOKENS = 5000      # enough for short CoT
    REQUEST_SLEEP = 1          # safe for 60 RPM

In [5]:
# =============================================================================
#  DATA LOADING & PREPARATION
# =============================================================================
FILENAME_PATTERN = re.compile(r"SupremeCourt_(\d{4})_(\d+)\.txt")

def parse_lrec_line(line: str):
    parts = line.strip().split("\t")
    fname_indices = [i for i, p in enumerate(parts) if p.endswith(".txt")]
    if len(fname_indices) != 2: return None
    try:
        sentpair_id = int(parts[0])
        sent1 = " ".join(parts[fname_indices[0] + 2 : fname_indices[1]]).strip('"')
        sent2 = " ".join(parts[fname_indices[1] + 2 : len(parts) - 2]).strip('"')
        label = parts[-1]
    except (ValueError, IndexError): return None
    return {"id": sentpair_id, "sent1": sent1, "sent2": sent2, "label": label}

def load_lrec_file(filepath: str) -> pd.DataFrame:
    rows = []
    print(f"Reading data from {filepath}...")
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            if parsed_row := parse_lrec_line(line):
                rows.append(parsed_row)
    return pd.DataFrame(rows)

def load_and_split_data(config: Config):
    """
    Returns:
      df_test: test dataframe
      df_shots: pool from which few-shot exemplars are drawn (disjoint from df_test)
    """
    print("Loading and splitting data...")
    df_pos = load_lrec_file(config.POS_DATA_FILE)
    df_neg = load_lrec_file(config.NEG_DATA_FILE)
    if df_pos.empty or df_neg.empty:
        print("ERROR: Data files failed to load.")
        return None, None

    # Few-shot pool (sample a chunk to keep memory & speed reasonable)
    df_pos_pool = df_pos.sample(n=min(10000, len(df_pos)), random_state=config.RANDOM_SEED)
    df_neg_pool = df_neg.sample(n=min(10000, len(df_neg)), random_state=config.RANDOM_SEED)
    df_shots = pd.concat([df_pos_pool, df_neg_pool]).reset_index(drop=True)

    # Test set excludes the shot pool to avoid leakage
    remaining_pos = df_pos.drop(df_pos_pool.index)
    remaining_neg = df_neg.drop(df_neg_pool.index)
    df_test = pd.concat([remaining_pos, remaining_neg]).reset_index(drop=True)
    df_test = df_test.dropna(subset=['id', 'sent1', 'sent2', 'label'])
    print(f"Created test set with {len(df_test)} samples.")

    if config.SAMPLE_SIZE > 0:
        print(f"Using a sample of {config.SAMPLE_SIZE} examples from the test set for evaluation.")
        df_test = df_test.sample(n=min(config.SAMPLE_SIZE, len(df_test)), random_state=config.RANDOM_SEED).reset_index(drop=True)

    return df_test, df_shots

In [6]:
FEWSHOT_COT_PROMPT = """
You are an expert legal analyst specializing in argumentation theory.

Your task is to classify the relationship between two sentences from Indian Supreme Court judgments.

Possible labels:
- SUPPORT: Sentence 2 strengthens or logically supports Sentence 1
- ATTACK: Sentence 2 weakens, rebuts, limits, or challenges Sentence 1
- NO_REL: The sentences are unrelated

For each example:
1. Give a short legal reasoning (2–3 sentences)
2. Output the final label in the exact format:
   Final Label: <SUPPORT|ATTACK|NO_REL>

---
Example 1:
Sentence 1: "The Tribunal misdirected itself in law."
Sentence 2: "The appeal is therefore allowed and remanded."
Reasoning: The second sentence states the legal consequence of the error identified in the first sentence. It follows directly from the finding of misdirection.
Final Label: SUPPORT

---
Example 2:
Sentence 1: "The conviction was based on unlawful assembly."
Sentence 2: "The accused was acquitted under Section 148 IPC."
Reasoning: The second sentence negates the factual basis required for the conviction mentioned in the first sentence. This undermines the original legal conclusion.
Final Label: ATTACK

---
Example 3:
Sentence 1: "The defendant was guilty of perjury."
Sentence 2: "The Companies Act amendment comes into force next month."
Reasoning: The two sentences concern unrelated legal topics and do not affect each other.
Final Label: NO_REL

---
Now analyze the following pair.

Sentence 1: "{sent1}"
Sentence 2: "{sent2}"

Reasoning:
"""

In [7]:
def extract_openai_text_and_tokens(response):
    text = response.choices[0].message.content.strip()

    usage = response.usage
    tokens = {
        "prompt_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens,
    }

    return text, tokens

In [8]:
class OpenAIFewShotCoTClassifier:
    def __init__(self, config: Config):
        self.config = config

    def parse_output(self, text: str) -> str:
        text = text.upper()
        if "FINAL LABEL:" in text:
            label = text.split("FINAL LABEL:")[-1].strip().split()[0]
            if label in {"SUPPORT", "ATTACK", "NO_REL"}:
                return label
        return "NO_REL"

    def predict(self, df: pd.DataFrame):
        predictions = []
        prompt_tokens = []
        output_tokens = []
        total_tokens = []
        reasons = []

        for _, row in tqdm(df.iterrows(), total=len(df)):
            prompt = FEWSHOT_COT_PROMPT.format(
                sent1=row.sent1,
                sent2=row.sent2
            )

            try:
                response = client.chat.completions.create(
                    model=self.config.MODEL_NAME,
                    messages=[
                        {"role": "system", "content": "You classify legal sentence relations."},
                        {"role": "user", "content": prompt},
                    ],
                    temperature=self.config.TEMPERATURE,
                    max_tokens=self.config.MAX_OUTPUT_TOKENS,
                )

                text, usage = extract_openai_text_and_tokens(response)
                pred = self.parse_output(text)

            except Exception as e:
                print("Error in FewShot CoT:", e)
                pred = "NO_REL"
                text = ""
                usage = {"prompt_tokens": 0, "output_tokens": 0, "total_tokens": 0}

            predictions.append(pred)
            prompt_tokens.append(usage["prompt_tokens"])
            output_tokens.append(usage["output_tokens"])
            total_tokens.append(usage["total_tokens"])
            reasons.append(text)

            time.sleep(self.config.REQUEST_SLEEP)

        return predictions, prompt_tokens, output_tokens, total_tokens, reasons

In [9]:
if __name__ == "__main__":
    config = Config()
    df_test, _ = load_and_split_data(config)

    clf = OpenAIFewShotCoTClassifier(config)
    preds, p_tok, o_tok, t_tok, reasons = clf.predict(df_test)

    true_labels = (
        df_test["label"]
        .str.upper()
        .str.replace(" ", "_")
        .replace({"NO_RELATION": "NO_REL"})
    )

    print("Accuracy:", accuracy_score(true_labels, preds))
    print(classification_report(true_labels, preds, zero_division=0))

    out = pd.DataFrame({
        "pair_id": df_test["id"],
        "sentence_1": df_test["sent1"],
        "sentence_2": df_test["sent2"],
        "true_label": df_test["label"],
        "predicted_label": preds,
        "prompt_token": p_tok,
        "output_token": o_tok,
        "total_token": t_tok,
        "reason": reasons
    })

    out.to_csv(config.OUTPUT_FILE, index=False)
    print(f"Saved results to {config.OUTPUT_FILE}")

Loading and splitting data...
Reading data from /kaggle/input/lrec-tcs-attack-support/sentencePair.txt...
Reading data from /kaggle/input/lrec-tcs-attack-support/sentencePair_neg.txt...
Created test set with 20506 samples.
Using a sample of 500 examples from the test set for evaluation.


100%|██████████| 500/500 [20:06<00:00,  2.41s/it]

Accuracy: 0.58
              precision    recall  f1-score   support

      ATTACK       0.51      0.49      0.50       130
      NO_REL       0.90      0.53      0.67       247
     SUPPORT       0.41      0.77      0.54       123

    accuracy                           0.58       500
   macro avg       0.61      0.60      0.57       500
weighted avg       0.68      0.58      0.59       500

Saved results to /kaggle/working/openai_fewshot_cot.csv
